# CNN 수업 복습 응용
이번 수업 복습에는 2가지 목적이 있습니다.
- 수업에서 다룬 코드 템플릿을 따라가면서 보지 않고도 원활하게 구조를 설계하고 코드를 작성할 수 있게 되는 것
- 함수를 설계하며 파이토치 내부적인 구조 등에 대해서 이해하며 여러 기법 등에 대한 사고를 할 수 있게 되는 것

수업에서는 다음과 같은 내용을 학습했습니다.

| 영역         | 수업에서 나온 내용               | 코드/객체                                                |
| ---------- | ------------------------ | ---------------------------------------------------- |
| 환경         | 난수 고정                    | `random.seed`, `np.random.seed`, `torch.manual_seed` |
| 파일         | 경로 관리                    | `Path`                                               |
| 파일         | zip 데이터 압축 해제            | `zipfile.ZipFile`, `extractall()`                    |
| Dataset    | 폴더 기반 이미지 Dataset        | `datasets.ImageFolder`                               |
| Dataset    | 클래스 정보                   | `.classes`, `.class_to_idx`, `.targets`              |
| 전처리        | 여러 변환 묶기                 | `transforms.Compose`                                 |
| 전처리        | 이미지 크기 통일                | `Resize`                                             |
| 전처리        | 1채널 변환                   | `Grayscale`                                          |
| 전처리        | 픽셀 반전                    | `RandomInvert`                                       |
| 증강         | 이동·회전·크기 변화              | `RandomAffine`                                       |
| Tensor     | 이미지 → Tensor             | `ToTensor`                                           |
| Scaling    | `[0,1] → [-1,1]` 근처      | `Normalize`                                          |
| 데이터 분리     | 클래스별 train/validation 분할 | `.targets`, `np.where`, `shuffle`                    |
| Dataset    | 일부 index만 사용             | `Subset`                                             |
| DataLoader | batch 생성                 | `DataLoader`                                         |
| DataLoader | 데이터 섞기                   | `shuffle`                                            |
| DataLoader | 로딩 worker                | `num_workers`                                        |
| 메모리        | pinned memory            | `pin_memory=True`                                    |
| Tensor     | CNN 입력 shape             | `[B, C, H, W]`                                       |
| 시각화        | 이미지 batch 확인             | `matplotlib`, `subplots`                             |
| CNN        | 합성곱                      | `Conv2d`                                             |
| CNN        | 채널 정규화                   | `BatchNorm2d`                                        |
| CNN        | 활성화                      | `ReLU(inplace=True)`                                 |
| CNN        | 공간 크기 축소                 | `MaxPool2d`                                          |
| CNN        | 출력 크기 강제 통일              | `AdaptiveAvgPool2d`                                  |
| CNN        | 벡터화                      | `Flatten`                                            |
| 규제         | 일부 activation 제거         | `Dropout`                                            |
| 분류         | FC 출력                    | `Linear`                                             |
| 모델         | Layer 묶기                 | `nn.Sequential`                                      |
| 장치         | CUDA/MPS/CPU 선택          | `torch.cuda`, `torch.backends.mps`                   |
| Loss       | 다중 분류 손실                 | `CrossEntropyLoss`                                   |
| Optimizer  | Adam + weight decay      | `AdamW`                                              |
| Scheduler  | 성능 정체 시 LR 감소            | `ReduceLROnPlateau`                                  |
| 학습모드       | train/eval 전환            | `model.train()`, `model.eval()`                      |
| Autograd   | 학습 시 grad 활성화            | `torch.enable_grad()`                                |
| 추론         | grad/추론 bookkeeping 제거   | `torch.inference_mode()`                             |
| CPU→GPU    | Tensor 장치 이동             | `.to(device)`                                        |
| 전송 최적화     | non-blocking 전송 요청       | `non_blocking=True`                                  |
| gradient   | gradient 초기화 최적화         | `zero_grad(set_to_none=True)`                        |
| 역전파        | gradient 계산              | `loss.backward()`                                    |
| 업데이트       | parameter 수정             | `optimizer.step()`                                   |
| 평가         | batch accuracy 누적        | `argmax`, `.sum()`                                   |
| 평가         | sample 수 기반 loss 평균      | `loss.item() * batch_size`                           |
| 학습관리       | loss/accuracy 기록         | `history`                                            |
| 학습관리       | best validation loss     | `best_val_loss`                                      |
| 학습관리       | Early Stopping 준비        | `patience`, `wait`                                   |
| 저장         | 모델 저장 경로                 | `.pth`                                               |


# 이번에 작업할 주제
이번에는 **위성사진 토지 종류 분류** 데이터셋을 이용하여

EuroSAT에서 제공해주는 `forest`, `river`, `residential` 등을 분류할 수 있게 하려고 합니다.

이를 통해서 색/질감/공간 패턴을 어떻게 보는지 확인하며 프레임워크 구조를 이해하는 것을 목표로 하겠습니다.

In [47]:
from operator import contains

import torch
from torchvision import datasets, transforms
from collections import Counter

import numpy as np
import pprint

from sklearn.model_selection import train_test_split

print(torch.__version__)
print(torch.cuda.is_available())

2.13.0+cu130
True


In [17]:
# torchvision.datasets를 이용하여
# EuroSAT 데이터셋을 불러옵니다.
dataset = datasets.EuroSAT(
    # 인터프리터가 현재 작업중인 경로 기준으로 계산되는 저장 경로입니다.
    root="./data",
    # 로컬에 데이터가 없으면 다운을 받겠다는 의미이며
    # 메모리에 저장을 하지 않는다는 것입니다.
    download=True
)

In [18]:
print(type(dataset))
print(isinstance(dataset, torch.utils.data.Dataset))
print(dataset.classes)
print(dataset.class_to_idx)

<class 'torchvision.datasets.eurosat.EuroSAT'>
True
['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
{'AnnualCrop': 0, 'Forest': 1, 'HerbaceousVegetation': 2, 'Highway': 3, 'Industrial': 4, 'Pasture': 5, 'PermanentCrop': 6, 'Residential': 7, 'River': 8, 'SeaLake': 9}


# 데이터셋 살펴보기
현재 받은 데이터셋은 `torchvision.datasets.eurosat.EuroSAT` 타입으로 `torch.utils.data.Dataset`을 상속한 클래스입니다.

이는 `__getitem__`을 통해 `tuple(PIL.Image.Image, int)` 데이터를  받을 수 있으며 첫번째 인자는 이미지 객체로 나타납니다. 해당 데이터의 사이즈를 찍으면 `(64, 64)`임을 확인할 수 있습니다.

In [19]:
image, label = dataset[0]

print(type(image))
print(isinstance(image, torch.Tensor))
print(type(label))
print(image.size)

<class 'PIL.Image.Image'>
False
<class 'int'>
(64, 64)


# PIL 이미지를 CNN이 받을 수 있는 Tensor로 바꾸고, 변환 전후로 직접 확인하기

해당 데이터를 받는 시점에 Dataset 클래스의 `__getitem__`의 transform을 등록하게 하여 `ToTensor()`을 적용시켜 줍니다.

여기에서 ToTensor은 Image를 Tensor 타입으로 변환하는 것 이외에도 0~255 이미지 픽셀값을 0.0 ~ 1.0 범위로 float32 Tensor로 변환시켜줍니다.

또한 데이터 하나의 `shape=(C, H, W)` 형태로 변환해주게 됩니다.

In [26]:
# 데이터를 Tensor 타입으로 바꾸며 정규화(standardization) 해주는 callable 객체 생성
transform = transforms.Compose([
    transforms.ToTensor(),
])

dataset = datasets.EuroSAT(
    root="./data",
    download=False,
    # 데이터에서 __getitem__을 시점에 transform을 적용하게 만들기
    transform=transform
)

# 2만 7천개의 데이터셋
print(len(dataset))

27000


In [22]:
image, label = dataset[0]

print(type(image))
print(isinstance(image, torch.Tensor))

print(image.shape)
print(image.dtype)

print(image.min())
print(image.max())

print("\n", image)

<class 'torch.Tensor'>
True
torch.Size([3, 64, 64])
torch.float32
tensor(0.3216)
tensor(0.8000)

 tensor([[[0.5843, 0.5843, 0.5804,  ..., 0.4000, 0.4078, 0.4039],
         [0.5843, 0.5843, 0.5804,  ..., 0.4000, 0.4078, 0.4039],
         [0.5608, 0.5569, 0.5725,  ..., 0.4000, 0.4000, 0.4039],
         ...,
         [0.5647, 0.5529, 0.5373,  ..., 0.4039, 0.3922, 0.3961],
         [0.5176, 0.5176, 0.4980,  ..., 0.4118, 0.4118, 0.4196],
         [0.4863, 0.4784, 0.4706,  ..., 0.4157, 0.4157, 0.4196]],

        [[0.4745, 0.4745, 0.4667,  ..., 0.3529, 0.3608, 0.3569],
         [0.4745, 0.4745, 0.4667,  ..., 0.3529, 0.3608, 0.3569],
         [0.4588, 0.4549, 0.4588,  ..., 0.3608, 0.3529, 0.3569],
         ...,
         [0.4549, 0.4588, 0.4588,  ..., 0.3647, 0.3529, 0.3569],
         [0.4314, 0.4392, 0.4314,  ..., 0.3686, 0.3647, 0.3725],
         [0.4118, 0.4157, 0.4039,  ..., 0.3725, 0.3686, 0.3725]],

        [[0.4706, 0.4706, 0.4745,  ..., 0.3922, 0.4000, 0.3961],
         [0.4706, 0.4706,

In [27]:
print("전체 데이터 수:", len(dataset))
print("클래스 수:", len(dataset.classes))
print("클래스: ", len(dataset.classes))
print("매핑:", dataset.class_to_idx)

전체 데이터 수: 27000
클래스 수: 10
클래스:  10
매핑: {'AnnualCrop': 0, 'Forest': 1, 'HerbaceousVegetation': 2, 'Highway': 3, 'Industrial': 4, 'Pasture': 5, 'PermanentCrop': 6, 'Residential': 7, 'River': 8, 'SeaLake': 9}


EuroSAT는 ImageFolder 계열이기 때문에 targets에 각 이미지의 정답 인덱스가 들어있습니다.

In [31]:
print(type(dataset.targets))
print(dataset.targets[:10])
print(dataset.targets[4500:4510])

<class 'list'>
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [43]:
# Counter 타입으로 dataset.targets - python list를 변환
class_counts = Counter(dataset.targets)

pprint.pprint(class_counts)

Counter({0: 3000,
         1: 3000,
         2: 3000,
         7: 3000,
         9: 3000,
         3: 2500,
         4: 2500,
         6: 2500,
         8: 2500,
         5: 2000})


In [41]:
eurosat_idx_to_class = {}

print("class_idx type:", type(list(dataset.class_to_idx)[0]))
print("class_name type:", type(list(dataset.class_to_idx)[1]))

for class_name,class_idx in dataset.class_to_idx.items():
    eurosat_idx_to_class[class_idx] = class_name

class_idx type: <class 'str'>
class_name type: <class 'str'>


# 층화 검증 데이터 분할하기
해당 데이터는 거의 3000~2000개로 적절한 비율을 유지한다고 판단되어 데이터를 비율대로 분리하겠습니다.

이때 사용하는 라이브러리는 skleanr.model_selection의 train_test_split입니다.

In [45]:
# 데이터를 그대로 메모리에 올릴 수 없기 때문에 인덱스를 기준으로 나눠줍니다.
indices = np.arange(len(dataset))

print(indices[:10])
print(len(indices))

[0 1 2 3 4 5 6 7 8 9]
27000


In [46]:
dataset

Dataset EuroSAT
    Number of datapoints: 27000
    Root location: ./data
    StandardTransform
Transform: Compose(
               ToTensor()
           )

In [73]:
# DatasetFolder 타입을 ndarray 형태로 변환하기 ->
targets = np.array(dataset.targets)

print(targets.shape)

train_idx, test_idx = train_test_split(
    # 해당 데이터에는 `indexable` 자료구조가 들어갈 수 있어서 ndarray도 들어갈 수 있습니다.
    indices,
    test_size=0.3,
    random_state=42,
    stratify=targets
)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.3,
    random_state=42,
    stratify=targets[train_idx]
)

len(train_idx), len(val_idx), len(test_idx)

(27000,)


(13230, 5670, 8100)

# Subset
Subset는 torch.utils.data.Subset 위치에 존재하며 첫번째 인자로 Dataset 타입을 받으며 두번째 인자로 해당 데이터의 인덱스 위치 리스트를 받아서 데이터 래퍼를 생성해줍니다.

복사 방식이 아니며 주소를 참조하여 사용하게 됩니다.

In [74]:
from torch.utils.data import Subset

# Subset 만들기
train_dataset = Subset(dataset, train_idx)
test_dataset = Subset(dataset, test_idx)

In [75]:
print(train_dataset)
print(len(train_dataset))
print(len(test_dataset))
# Subset끼리의 합연산이 성립한다.
print(len(train_dataset + test_dataset))
print(len(train_dataset[0]))
print(type(train_dataset[0][0]))
print(type(train_dataset[0][1]))
print(train_dataset[0][0].shape)

13230
8100
21330
2
<class 'torch.Tensor'>
<class 'int'>
torch.Size([3, 64, 64])


In [79]:
# 원본 데이터셋
print(train_dataset.dataset)
print(train_dataset.dataset is dataset)
# 실제로 사용하는 식별자들
# 실제로 indices[0] = 0이 아닐 수 있습니다.
# train_dataset[0] = train_dataset.dataset[train_dataset.indices[0]] 입니다. (개념상)
print(train_dataset.indices[:10])

Dataset EuroSAT
    Number of datapoints: 27000
    Root location: ./data
    StandardTransform
Transform: Compose(
               ToTensor()
           )
True
[23822 10937 25770  2691 14643  5710  5124  3223 22157  3563]


In [ ]:
# transform 파이프라인 만들어주기
train_transform = transforms.Compose([
    # 좌우 대칭
    transforms.RandomHorizontalFlip(),
    # 수직 대칭
    transforms.RandomVerticalFlip(),
    # 정규화 + 텐서화
    transforms.ToTensor()
])

# 테스트할 때에는 그대로 데이터 형태만 Tensor로 바꿔주면 됨
eval_transform = transforms.Compose([
    transforms.ToTensor()
])


In [81]:
train_full = datasets.EuroSAT(
    root="./data",
    download=False,
    transform=train_transform
)

eval_full = datasets.EuroSAT(
    root="./data",
    download=False,
    transform=eval_transform
)

In [82]:
train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(eval_full, val_idx)
test_dataset = Subset(eval_full, test_idx)

In [92]:
# transform을 거쳐서 데이터가 변환되어 잘못 나올 수 있음
flags = []
while len(flags) < 2:
    img1, label1 = train_dataset[0]
    img2, label2 = train_dataset[0]
    flag =torch.equal(img1, img2)
    print(flag)
    if flags.__contains__(flag):
        continue
    else:
        flags.append(flag)

True
True
False


In [96]:
flags = []

for i in range(100):
    img1, _ = val_dataset[0]
    img2, _ = val_dataset[0]

    flag = torch.equal(img1, img2)

    if not flags.__contains__(flag):
        flags.append(flag)
        print(f"{flag} appended", flags)

    if len(flags) > 1:
        print("both occured")
        break

print(flags)


True appended [True]
[True]
